In [25]:
library(ggplot2)
library(dplyr)
# Load the required libraries
library(ggpubr)
# load dplyr and tidyr library

library(tidyr)
library(parallel)
library(future)
library(furrr)

In [26]:
setwd("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/analysis/sRNA_deseq/genric_regions")

In [27]:
data <- read.csv(gzfile("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/STAR_uniqe_marker_strain_wise/fetureCountlist3/all_list3_uniq_piRNa.tab.gz"), header=FALSE,sep='\t')
data

V1,V2,V3,V4,V5,V6,V7,V8
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<chr>
exon:None_6972170_6972318,129S1_SvImJ#1#chr1,6972170,6972318,+,149,0.16,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7294445,129S1_SvImJ#1#chr1;129S1_SvImJ#1#chr1,7292335;7292335,7294445;7294445,+;+,2111,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7294372,129S1_SvImJ#1#chr1,7292335,7294372,+,2038,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7294447,129S1_SvImJ#1#chr1,7292335,7294447,+,2113,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7293220,129S1_SvImJ#1#chr1,7292335,7293220,+,886,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7293610,129S1_SvImJ#1#chr1,7292335,7293610,+,1276,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7293455,129S1_SvImJ#1#chr1,7292335,7293455,+,1121,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_13924083_13924230,129S1_SvImJ#1#chr1,13924083,13924230,+,148,12.16,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_18403597_18408662,129S1_SvImJ#1#chr1,18403597,18408662,+,5066,0.33,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts


In [28]:
data <- data %>% filter(V7 != 0) 
data <- data %>% separate(V1, c('V9', 'V10','V11','V12'), sep =" ") 
data


Warning message:
“Expected 4 pieces. Missing pieces filled with `NA` in 1386964 rows [1, 2, 3, 4,
5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...].”


V9,V10,V11,V12,V2,V3,V4,V5,V6,V7,V8
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<chr>
exon:None_6972170_6972318,NA,NA,NA,129S1_SvImJ#1#chr1,6972170,6972318,+,149,0.16,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7294445,NA,NA,NA,129S1_SvImJ#1#chr1;129S1_SvImJ#1#chr1,7292335;7292335,7294445;7294445,+;+,2111,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7294372,NA,NA,NA,129S1_SvImJ#1#chr1,7292335,7294372,+,2038,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7294447,NA,NA,NA,129S1_SvImJ#1#chr1,7292335,7294447,+,2113,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7293220,NA,NA,NA,129S1_SvImJ#1#chr1,7292335,7293220,+,886,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7293610,NA,NA,NA,129S1_SvImJ#1#chr1,7292335,7293610,+,1276,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_7292335_7293455,NA,NA,NA,129S1_SvImJ#1#chr1,7292335,7293455,+,1121,0.05,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_13924083_13924230,NA,NA,NA,129S1_SvImJ#1#chr1,13924083,13924230,+,148,12.16,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts
exon:None_18403597_18408662,NA,NA,NA,129S1_SvImJ#1#chr1,18403597,18408662,+,5066,0.33,129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts


In [29]:
# Group by V8, V9, V10 and sum the V7 values
new_data <- data %>%
  group_by(V8, V9) %>%
  summarise(summed_V7 = sum(V7, na.rm = TRUE)) %>%
  ungroup()
new_data

write.csv(new_data, "uniq_piRNA_list3_count.csv", row.names=FALSE)

`summarise()` has grouped output by 'V8'. You can override using the `.groups`
argument.


V8,V9,summed_V7
<chr>,<chr>,<dbl>
129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts,exon:None_100123244_100127936,0.33
129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts,exon:None_100525456_100525926,2.00
129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts,exon:None_100554702_100559377,0.17
129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts,exon:None_101134435_101137259,0.40
129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts,exon:None_101407742_101411557,0.17
129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts,exon:None_10189444_10192446,1.05
129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts,exon:None_10189444_10192610,1.05
129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts,exon:None_102018622_102020510,2.67
129S1_SvImJ/129S1_SvImJ-12.5dpp.1-P12.5.featureCounts,exon:None_102018622_102027055,2.67
